# Aphasia Recovery: Neural Network Maintenance Forecasting
### Objective
This analysis utilizes a **Neural Network (Artificial Intelligence)** to predict whether a word will be successfully named 1 month after treatment. Neural Networks excel at finding complex, non-linear relationships in clinical data.

## 🧠 Neural Network Architecture
This model simulates the way a brain processes information through layers of 'neurons'.

### 1. The Inputs (Input Layer)
The model receives 9 clinical and behavioral features, including Learning Velocity, Success Streaks, and Patient Demographics. All inputs are **standardized** to ensure the network remains stable.

### 2. The Hidden Layer (100 Neurons)
A layer of 100 artificial neurons connects the inputs to the output. These neurons learn the complex weights—for example, how a high 'MPO' (Time since stroke) might interact specifically with high 'RT Jitter' (Instability) to predict failure.

### 3. The Activation (ReLU)
We use the 'Rectified Linear Unit' activation function, which allows the network to learn non-linear recovery patterns that simpler models might miss.

### 4. The Output (Prediction)
The output is a probability score between 0 and 1, where 1 indicates successful word maintenance at the follow-up phase.

## 🔍 Deep Dive: How we calculate 'Learning Velocity'
Learning velocity captures the **momentum** of recovery, moving beyond static counts.

### The Formulaic Logic:
1. **Grouping**: We group data by `player` and `word` to track individual trajectories.
2. **Rolling Accuracy**: We calculate a **3-trial rolling average**. 
3. **The Velocity (Slope)**: We calculate the difference between the current baseline and the previous one. 
   * **Positive (+)**: The patient is actively acquiring the word.
   * **Zero (0)**: Performance has plateaued (either mastered or stuck).
   * **Negative (-)**: Performance is regressing (retrieval is becoming harder).

## 1. Data Integration & Engineering
In this step, we merge 5 clinical datasets into a single 'Timeline'. Every line is commented for full transparency.

In [ ]:
# Import deep learning and data processing libraries
import pandas as pd # Dataframes
import numpy as np  # Math
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.neural_network import MLPClassifier # THE NEURAL NETWORK ENGINE
from sklearn.preprocessing import StandardScaler # REQUIRED FOR NEURAL NETWORKS
from sklearn.metrics import classification_report, confusion_matrix, precision_score, recall_score, f1_score

# Plot aesthetics
sns.set_theme(style='whitegrid', context='talk')
plt.rcParams['figure.figsize'] = (12, 6)

# LOAD DATASETS
probes = pd.read_csv('1. 2019-11-12_naming_probes_tidy.csv')
tx_nop = pd.read_csv('2. 2019-10-25_treatment_retrieval_noprime.csv')
tx_p = pd.read_csv('4. 2019-10-25_treatment_retrieval_prime.csv')
outcomes = pd.read_csv('6. 2019-10-3_Outcome_data_tidy.csv')
rewards = pd.read_csv('5. 2019-12-3_coins-stars.csv')

# STANDARDIZE PATIENT NAMES
for d in [probes, tx_nop, tx_p, outcomes, rewards]:
    d['player'] = d['player'].str.lower()

# CLEAN AND PREP TIMELINE
p_clean = probes[['player', 'date', 'session', 'target', 'category', 'trial_resp.corr.hand', 'vocal.rt', 'complexity_score', 'phase']].rename(columns={'target':'word', 'trial_resp.corr.hand':'success'})
nop_clean = tx_nop[['player', 'date', 'session', 'stim_text', 'category', 'naming1_resp.corr', 'naming1_vocal.rt']].rename(columns={'stim_text':'word', 'naming1_resp.corr':'success', 'naming1_vocal.rt':'vocal.rt'})
p_clean_tx = tx_p[['player', 'date', 'session', 'stim_text', 'category', 'naming2_resp.corr', 'naming2_vocal.rt']].rename(columns={'stim_text':'word', 'naming2_resp.corr':'success', 'naming2_vocal.rt':'vocal.rt'})

df = pd.concat([p_clean, nop_clean, p_clean_tx])
df['date'] = pd.to_datetime(df['date'].str.replace('_', '-'), errors='coerce', format='mixed')
df['phase'] = df['phase'].fillna('treatment')

# MERGE CLINICAL MARKERS
demos = outcomes.groupby('player').agg({'demo.mpo': 'first', 'demo.age': 'first'}).reset_index()
df = df.merge(demos, on='player', how='left')
reward_agg = rewards.groupby(['player', 'session', 'category']).agg({'stars': 'sum', 'coins': 'sum'}).reset_index()
df = df.merge(reward_agg, on=['player', 'session', 'category'], how='left').fillna({'stars': 0, 'coins': 0})
comp_map = probes.groupby('target')['complexity_score'].mean()
df['complexity_score'] = df['complexity_score'].fillna(df['word'].map(comp_map))

# --- ADVANCED FEATURE ENGINEERING ---
df['rolling_acc'] = df.groupby(['player', 'word'])['success'].transform(lambda x: x.rolling(window=3, min_periods=1).mean())
df['learning_velocity'] = df.groupby(['player', 'word'])['rolling_acc'].diff().fillna(0)
df['rt_jitter'] = df.groupby(['player', 'word'])['vocal.rt'].transform(lambda x: x.rolling(window=5, min_periods=1).std() / (x.rolling(window=5, min_periods=1).mean() + 1e-9)).fillna(0)
def get_streak(x):
    y = x.cumsum()
    return y.sub(y.mask(x != 0).ffill().fillna(0))
df['consecutive_success'] = df.groupby(['player', 'word'])['success'].transform(get_streak).shift(1).fillna(0)
df['rt_efficiency'] = df['vocal.rt'] / (df['complexity_score'] + 1e-9)
df['s'] = df.groupby(['player', 'word'])['success'].shift(1).fillna(0).groupby([df['player'], df['word']]).cumsum()

df = df.sort_values(['player', 'word', 'date']).dropna(subset=['success', 'complexity_score', 'demo.mpo'])
print(f'Pipeline integration complete: {len(df)} trials.')

## 2. Neural Network Construction
Training the **Multi-Layer Perceptron (MLP)** on the treatment dataset.

In [ ]:
# Define predictors for the Neural Network
features = ['s', 'consecutive_success', 'learning_velocity', 'rt_jitter', 'rt_efficiency', 'complexity_score', 'demo.mpo', 'demo.age', 'stars']
df_model = df.fillna(0)

# TEMPORAL SPLIT (Predicting the future 1-month follow-up)
train = df_model[df_model['phase'] != 'followup']
test = df_model[df_model['phase'] == 'followup']
X_train, y_train = train[features], train['success']
X_test, y_test = test[features], test['success']

# STEP 1: NEURAL SCALING
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# STEP 2: BUILD AND TRAIN THE NEURAL NETWORK
model = MLPClassifier(hidden_layer_sizes=(100,), max_iter=1000, random_state=42, activation='relu')
model.fit(X_train_scaled, y_train)
print('Neural Network (MLP) training complete.')

## 3. Post-Neural Performance Metrics
Accuracy, precision, recall, and F1-score for the follow-up trials predicted by the neural engine.

In [ ]:
# Neural predictions on scaled test data
y_pred = model.predict(X_test_scaled)
cm = confusion_matrix(y_test, y_pred)

# CALCULATE FINAL SCORES
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
accuracy = model.score(X_test_scaled, y_test)

metrics_df = pd.DataFrame({
    'Metric': ['Accuracy', 'Precision', 'Recall', 'F1-Score'],
    'Score': [f'{accuracy:.4f}', f'{precision:.4f}', f'{recall:.4f}', f'{f1:.4f}']
})
display(metrics_df)

# VISUALIZE THE NEURAL CONFUSION MATRIX (COUNTS & PERCENTAGES)
plt.figure(figsize=(16, 6))
plt.subplot(1, 2, 1)
sns.heatmap(cm, annot=True, fmt='d', cmap='RdPu', cbar=False)
plt.title('Neural Network: Success Counts (Confusion Matrix)')
plt.xlabel('Predicted Outcome (1=Success)')
plt.ylabel('Actual Outcome (1=Success)')

plt.subplot(1, 2, 2)
cm_perc = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
sns.heatmap(cm_perc, annot=True, fmt='.1%', cmap='RdPu', cbar=False)
plt.title('Neural Network: Success Percentage (Normalized)')
plt.xlabel('Predicted Outcome (1=Success)')
plt.ylabel('Actual Outcome (1=Success)')
plt.tight_layout()
plt.show()